# rizzvision-v2: Multi-class Clothing Classifier (tops / bottoms / other)

**Setup:** Mount Drive in Cell 1, enable T4 GPU (Runtime → Change runtime type), run all cells.

**Outputs:** `clothing_classifier_v2.keras` + `thresholds_v2.json` saved to Drive and downloaded.

**Changes from v1:** 3-class softmax head instead of binary sigmoid; per-class threshold calibration; stronger augmentation pipeline.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/rizzvision_v2'
DATASET_DIR = '/content/clothing_dataset'
OUT_DIR     = '/content'

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted. Output will be mirrored to:', DRIVE_DIR)

In [ ]:
import os, json, math
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report
import shutil

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('Contents:', os.listdir(DATASET_DIR))

INPUT_SIZE = 300
BATCH_SIZE = 32  # increase to 48/64 if using A100/L4
CLASSES    = ['tops', 'bottoms', 'other']
NUM_CLASSES = len(CLASSES)

In [ ]:
# ── Dataset pipeline ──────────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 40.0)
    image = tf.image.random_contrast(image, 0.75, 1.25)
    image = tf.image.random_saturation(image, 0.75, 1.25)
    image = tf.image.random_crop(image, [int(INPUT_SIZE * 0.9), int(INPUT_SIZE * 0.9), 3])
    image = tf.image.resize(image, [INPUT_SIZE, INPUT_SIZE])
    image = tf.clip_by_value(image, 0, 255)
    return image, label

def make_dataset(split, augment_fn=None, shuffle=False):
    ds = keras.utils.image_dataset_from_directory(
        os.path.join(DATASET_DIR, split),
        class_names=CLASSES,
        image_size=(INPUT_SIZE, INPUT_SIZE),
        batch_size=BATCH_SIZE,
        label_mode='categorical',
        shuffle=shuffle,
        seed=42,
    )
    if augment_fn:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_dataset('train', augment_fn=augment, shuffle=True)
val_ds   = make_dataset('val')
test_ds  = make_dataset('test')
print('Datasets loaded. Class order:', CLASSES)

In [ ]:
# ── Build model ───────────────────────────────────────────────────────────────
def build_model(trainable_base=False):
    base = keras.applications.EfficientNetB3(
        include_top=False,
        weights='imagenet',
        input_shape=(INPUT_SIZE, INPUT_SIZE, 3),
    )
    base.trainable = trainable_base
    inputs = keras.Input(shape=(INPUT_SIZE, INPUT_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    # Softmax output — one score per class
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return keras.Model(inputs, outputs), base

model, base = build_model(trainable_base=False)
model.summary()

In [ ]:
# ── Drive checkpoint callback ─────────────────────────────────────────────────
class DriveCheckpoint(keras.callbacks.Callback):
    def __init__(self, src, dst):
        self.src = src; self.dst = dst
    def on_epoch_end(self, epoch, logs=None):
        try: shutil.copy2(self.src, self.dst)
        except: pass

drive_cb = DriveCheckpoint(
    f'{OUT_DIR}/clothing_classifier_v2.keras',
    f'{DRIVE_DIR}/clothing_classifier_v2.keras'
)

# ── Phase 1: train head only (5 epochs) ───────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc', multi_label=False)],
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            f'{OUT_DIR}/clothing_classifier_v2.keras',
            monitor='val_accuracy', mode='max',
            save_best_only=True, verbose=1,
        ),
        drive_cb,
    ],
)

In [ ]:
# ── Phase 2: fine-tune top-100 base layers (40 epochs) ───────────────────────
base.trainable = True
for layer in base.layers[:-100]:
    layer.trainable = False

PHASE2_EPOCHS = 40
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-4,
    decay_steps=PHASE2_EPOCHS * len(train_ds),
    alpha=1e-6,
)

model.compile(
    optimizer=keras.optimizers.Adam(lr_schedule),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc', multi_label=False)],
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            f'{OUT_DIR}/clothing_classifier_v2.keras',
            monitor='val_accuracy', mode='max',
            save_best_only=True, verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=6, mode='max',
            restore_best_weights=True, verbose=1,
        ),
        drive_cb,
    ],
)

In [ ]:
# ── Per-class threshold calibration on validation set ─────────────────────────
# For a multi-class model we pick the confidence floor per class
# below which we fall back to 'other' (unknown / unrecognised)

best_model = keras.models.load_model(f'{OUT_DIR}/clothing_classifier_v2.keras')

val_probs, val_labels = [], []
for batch_x, batch_y in val_ds:
    val_probs.extend(best_model.predict(batch_x, verbose=0).tolist())
    val_labels.extend(np.argmax(batch_y.numpy(), axis=1).tolist())

val_probs  = np.array(val_probs)   # shape (N, 3)
val_labels = np.array(val_labels)  # shape (N,)

# For each class, find the confidence threshold that gives >= 90% precision
thresholds = {}
for i, cls in enumerate(CLASSES):
    class_probs = val_probs[:, i]
    best_thresh = 0.5
    for thresh in np.arange(0.3, 0.95, 0.01):
        preds = (class_probs >= thresh)
        tp = ((preds == 1) & (val_labels == i)).sum()
        fp = ((preds == 1) & (val_labels != i)).sum()
        precision = tp / (tp + fp + 1e-9)
        if precision >= 0.90:
            best_thresh = float(thresh)
            break
    thresholds[cls] = round(best_thresh, 3)
    print(f'{cls}: threshold={best_thresh:.3f}')

thresholds_out = {'thresholds': thresholds, 'classes': CLASSES}
with open(f'{OUT_DIR}/thresholds_v2.json', 'w') as f:
    json.dump(thresholds_out, f, indent=2)
shutil.copy2(f'{OUT_DIR}/thresholds_v2.json', f'{DRIVE_DIR}/thresholds_v2.json')
print('Thresholds saved:', thresholds)

In [ ]:
# ── Test set evaluation ───────────────────────────────────────────────────────
test_probs, test_labels = [], []
for batch_x, batch_y in test_ds:
    test_probs.extend(best_model.predict(batch_x, verbose=0).tolist())
    test_labels.extend(np.argmax(batch_y.numpy(), axis=1).tolist())

test_probs  = np.array(test_probs)
test_labels = np.array(test_labels)
test_preds  = np.argmax(test_probs, axis=1)

accuracy = (test_preds == test_labels).mean()
print(f'\n=== Test Set Results ===')
print(f'Overall Accuracy: {accuracy:.4f} (target >= 0.93)')
print()
print('Per-class report:')
print(classification_report(test_labels, test_preds, target_names=CLASSES))
print()
print('Confusion matrix (rows=actual, cols=predicted):')
print(confusion_matrix(test_labels, test_preds))
print('Classes:', CLASSES)

In [ ]:
# ── Download outputs ──────────────────────────────────────────────────────────
from google.colab import files
files.download(f'{OUT_DIR}/clothing_classifier_v2.keras')
files.download(f'{OUT_DIR}/thresholds_v2.json')